# End-to-end Learning with Autoencoders over Rayleigh Fading with Estimated CSI

**Author:** Yuxing Mao — *AI-Native Transceiver Design for Next-Generation Wireless Networks*

Extends the Sionna autoencoder tutorial from AWGN to block-fading Rayleigh with pilot-based
channel estimation. Ordered by dependency: **all definitions first (Part I), all experiments
after (Part II)**. `Run All` works end to end.

## Findings at a glance

| # | Question | Answer | Section |
|---|---|---|---|
| 1 | Does the autoencoder beat 64-QAM under AWGN? | Yes — 0.33 dB, from constellation shaping | 8 |
| 2 | What does Rayleigh cost? | 2.54 dB vs AWGN (theory: 1.9 dB) | — |
| 3 | Can AWGN weights transfer to Rayleigh? | **No** — U-shaped BER from distribution shift | 11 |
| 4 | Does retraining fix it? | Yes — 3.5e-4 at 14 dB with *estimated* CSI | 12 |
| 5 | Optimal pilot count P? | SNR-dependent; curves cross near 11 dB | 13 |
| 6 | Does a CNN demapper help? | No — flat fading offers no symbol correlation | 14 |
| 7 | Is 128 hidden units too large? | Yes — 32 units (1/13 params) nearly matches | 15 |
| 8 | Where does 1e-5 come from? | LDPC coding gain; uncoded stays near 2e-2 | 16 |
| 9 | Does the training-SNR strategy matter? | No — N0 is an explicit input | 17 |

## Design invariants worth stating up front

- **CSI never reaches the transmitter.** `h` is generated *inside* the channel function, after
  the mapper has already produced `x`. The learned constellation is a global parameter, not a
  function of any realisation of `h`.
- **Information bits are i.i.d. uniform** (`BinarySource`, p = 0.5), 750 info / 1500 coded bits
  per codeword — not a small fixed payload.
- **Pilot overhead is charged for**: `coderate_eff = coderate * (L - P) / L`.


# Part I — Definitions

Definitions and imports only; nothing is computed here.

## 1. Imports and simulation parameters

In [ ]:
# Import Sionna
try:
    import sionna.phy
except ImportError as e:
    import os
    import sys
    if 'google.colab' in sys.modules:
       # Install Sionna in Google Colab
       print("Installing Sionna and restarting the runtime. Please run the cell again.")
       os.system("pip install sionna")
       os.kill(os.getpid(), 5)
    else:
       raise e

import pickle

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch._dynamo

from sionna.phy import Block
from sionna.phy.channel import AWGN
from sionna.phy.utils import ebnodb2no, expand_to_rank, sim_ber
from sionna.phy.fec.ldpc import LDPC5GEncoder, LDPC5GDecoder
from sionna.phy.mapping import Mapper, Demapper, Constellation, BinarySource

sionna.phy.config.seed = 42  # Set seed for reproducible random number generation

%matplotlib inline

In [ ]:
###############################################
# SNR range for evaluation and training [dB]
###############################################
ebno_db_min = 4.0
ebno_db_max = 8.0

###############################################
# Modulation and coding configuration
###############################################
num_bits_per_symbol = 6  # Baseline is 64-QAM
modulation_order = 2**num_bits_per_symbol
coderate = 0.5  # Coderate for the outer code
n = 1500  # Codeword length [bit]. Must be a multiple of num_bits_per_symbol
num_symbols_per_codeword = n // num_bits_per_symbol  # Number of modulated baseband symbols per codeword
k = int(n * coderate)  # Number of information bits per codeword

###############################################
# Training configuration
###############################################
num_training_iterations_conventional = 10000  # Number of training iterations for conventional training
# Number of training iterations with RL-based training for the alternating training phase and fine-tuning of the receiver phase
num_training_iterations_rl_alt = 7000
num_training_iterations_rl_finetuning = 3000
training_batch_size = 128  # Training batch size
rl_perturbation_var = 0.01  # Variance of the perturbation used for RL-based training of the transmitter
model_weights_path_conventional_training = "awgn_autoencoder_weights_conventional_training"  # Filename to save the autoencoder weights once conventional training is done
model_weights_path_rl_training = "awgn_autoencoder_weights_rl_training"  # Filename to save the autoencoder weights once RL-based training is done

###############################################
# Evaluation configuration
###############################################
results_filename = "awgn_autoencoder_results"  # Location to save the results

## 2. Neural demapper and end-to-end systems (MLP)

`NeuralDemapper` maps `[Re(y), Im(y), log10(N0)]` to one LLR **per bit**, and processes each
symbol **independently** — `nn.Linear` acts on the last axis only, so the 250 symbols of a
codeword never interact. This matches the optimal APP demapper, which is also per-symbol.

Layer sizes: input **3** (physics: a complex sample split into real/imag, plus noise power —
the same `y` implies completely different LLRs at different N0), hidden **128 x 2**
(tutorial default; see Section 15), output **6** (one LLR per bit of a 64-QAM symbol).
The `width` argument is parameterised for the capacity study.

In [ ]:
class NeuralDemapper(nn.Module):
    """Neural network-based demapper with three dense layers and ReLU activation.

    width: hidden layer size (default 128, the tutorial's original value).
    """

    def __init__(self, width=128):
        super().__init__()
        self._dense_1 = nn.Linear(3, width)
        self._dense_2 = nn.Linear(width, width)
        self._dense_3 = nn.Linear(width, num_bits_per_symbol)  # Output LLRs for every bit

    def forward(self, y, no):
        # Using log10 scale helps with the performance
        no_db = torch.log10(no)

        # AWGN passes no as [batch size, 1] and needs broadcasting.
        # Rayleigh passes a per-symbol no_eff that is already [batch size, num_symbols_per_codeword].
        if no_db.shape[1] == 1:
            no_db = no_db.expand(-1, num_symbols_per_codeword)

        z = torch.stack([y.real, y.imag, no_db], dim=2)

        llr = F.relu(self._dense_1(z))
        llr = F.relu(self._dense_2(llr))
        llr = self._dense_3(llr)

        return llr

In [ ]:
class E2ESystemConventionalTraining(nn.Module):
    """End-to-end communication system for conventional training.
    
    This system transmits bits modulated using a trainable constellation over
    an AWGN channel. The receiver uses a neural network-based demapper.
    """

    def __init__(self, training):
        super().__init__()
        
        self._training = training
        
        ################
        ## Transmitter
        ################
        self._binary_source = BinarySource()
        # To reduce the computational complexity of training, the outer code is not used when training
        if not self._training:
            # num_bits_per_symbol is required for the interleaver
            self._encoder = LDPC5GEncoder(k, n, num_bits_per_symbol)
        
        # Trainable constellation
        # We initialize a custom constellation with QAM points
        qam_points = Constellation("qam", num_bits_per_symbol).points
        self.points_r = nn.Parameter(qam_points.real.clone())
        self.points_i = nn.Parameter(qam_points.imag.clone())

        self.constellation = Constellation("custom",
                                           num_bits_per_symbol,
                                           points=torch.complex(self.points_r, self.points_i),
                                           normalize=True,
                                           center=True)
       
        
        self._mapper = Mapper(constellation=self.constellation)
        
        ################
        ## Channel
        ################
        self._channel = AWGN()
        
        ################
        ## Receiver
        ################
        # We use the previously defined neural network for demapping
        self._demapper = NeuralDemapper()
        # To reduce the computational complexity of training, the outer code is not used when training
        if not self._training:
            self._decoder = LDPC5GDecoder(self._encoder, hard_out=True)

    def forward(self, batch_size, ebno_db):
        
        # Update constellation points from trainable parameters (creates fresh graph)
        self.constellation.points = torch.complex(self.points_r, self.points_i)

        # If `ebno_db` is a scalar, a tensor with shape [batch size] is created
        if ebno_db.dim() == 0:
            ebno_db = ebno_db.expand(batch_size)
        no = ebnodb2no(ebno_db, num_bits_per_symbol, coderate)
        no = expand_to_rank(no, 2)
        
        ################
        ## Transmitter
        ################
        # Outer coding is only performed if not training
        if self._training:
            c = self._binary_source([batch_size, n])
        else:
            b = self._binary_source([batch_size, k])
            c = self._encoder(b)
        # Modulation
        x = self._mapper(c)  # x [batch size, num_symbols_per_codeword]
        
        ################
        ## Channel
        ################
        y = self._channel(x, no)  # [batch size, num_symbols_per_codeword]
        
        ################
        ## Receiver
        ################
        llr = self._demapper(y, no)
        llr = llr.reshape(batch_size, n)
        # If training, outer decoding is not performed and the BCE is returned
        if self._training:
            loss = F.binary_cross_entropy_with_logits(llr, c)
            return loss
        else:
            # Outer decoding
            b_hat = self._decoder(llr)
            return b, b_hat  # Ground truth and reconstructed information bits returned for BER/BLER computation

In [ ]:
class E2ESystemRayleigh(E2ESystemConventionalTraining):
    """Same TX/RX, but over i.i.d. Rayleigh flat fading with perfect CSI."""

    def forward(self, batch_size, ebno_db):
        self.constellation.points = torch.complex(self.points_r, self.points_i)
        if ebno_db.dim() == 0:
            ebno_db = ebno_db.expand(batch_size)
        no = ebnodb2no(ebno_db, num_bits_per_symbol, coderate)
        no = expand_to_rank(no, 2)

        # Transmitter
        if self._training:
            c = self._binary_source([batch_size, n])
        else:
            b = self._binary_source([batch_size, k])
            c = self._encoder(b)
        x = self._mapper(c)

        # Channel: Rayleigh flat fading
        std = 0.5 ** 0.5
        h_r = torch.randn(x.shape, device=x.device, dtype=x.real.dtype) * std
        h_i = torch.randn(x.shape, device=x.device, dtype=x.real.dtype) * std
        h = torch.complex(h_r, h_i)
        y = self._channel(h * x, no)

        # Receiver: coherent equalization
        z = y / h
        no_eff = no / (h.real ** 2 + h.imag ** 2)

        llr = self._demapper(z, no_eff)
        llr = llr.reshape(batch_size, n)

        if self._training:
            loss = F.binary_cross_entropy_with_logits(llr, c)
            return loss
        else:
            b_hat = self._decoder(llr)
            return b, b_hat

## 3. Utilities — training loop, save / load weights

In [ ]:
def conventional_training(model):
    """Train the model using conventional SGD with backpropagation."""
    # Optimizer used to apply gradients
    optimizer = torch.optim.Adam(model.parameters())
    device = sionna.phy.config.device
    
    for i in range(num_training_iterations_conventional):
        optimizer.zero_grad()
        # Sampling a batch of SNRs
        ebno_db = torch.empty(training_batch_size, device=device).uniform_(ebno_db_min, ebno_db_max)
        # Forward pass
        loss = model(training_batch_size, ebno_db)
        # Computing and applying gradients
        loss.backward()
        optimizer.step()
        # Printing periodically the progress
        if i % 100 == 0:
            print(f'Iteration {i}/{num_training_iterations_conventional}  BCE: {loss.item():.4f}', end='\r')
    print()
    model.eval()
    optimizer.zero_grad(set_to_none=True)

In [ ]:
def save_weights(model, model_weights_path):
    m = getattr(model, "_orig_mod", model)
    torch.save(m.state_dict(), model_weights_path)

In [ ]:
# Utility function to load and set weights of a model
def load_weights(model, model_weights_path):
    """Load and set weights of a model."""
    device = sionna.phy.config.device
    state = torch.load(model_weights_path, map_location=device)
    model.load_state_dict(state, strict=False)
    # Update the constellation points from the loaded parameters
    model.constellation.points = torch.complex(model.points_r, model.points_i)

## 4. Classical baselines

64-QAM + Gray labelling + **APP demapping** + LDPC, with no trainable parameters.
Given `h`, the pair `(y/h, N0/|h|^2)` is a sufficient statistic, so `BaselineRayleigh` is the
*optimal* receiver for its channel — a strong reference, not a strawman.

In [ ]:
class Baseline(nn.Module):
    """Baseline system using QAM with Gray labeling and conventional demapping."""

    def __init__(self):
        super().__init__()
        
        ################
        ## Transmitter
        ################
        self._binary_source = BinarySource()
        self._encoder = LDPC5GEncoder(k, n, num_bits_per_symbol)
        constellation = Constellation("qam", num_bits_per_symbol)
        self.constellation = constellation
        self._mapper = Mapper(constellation=constellation)
        
        ################
        ## Channel
        ################
        self._channel = AWGN()
        
        ################
        ## Receiver
        ################
        self._demapper = Demapper("app", constellation=constellation)
        self._decoder = LDPC5GDecoder(self._encoder, hard_out=True)

    def forward(self, batch_size, ebno_db):
        # If `ebno_db` is a scalar, a tensor with shape [batch size] is created
        if ebno_db.dim() == 0:
            ebno_db = ebno_db.expand(batch_size)
        no = ebnodb2no(ebno_db, num_bits_per_symbol, coderate)
        no = expand_to_rank(no, 2)
        
        ################
        ## Transmitter
        ################
        b = self._binary_source([batch_size, k])
        c = self._encoder(b)
        # Modulation
        x = self._mapper(c)  # x [batch size, num_symbols_per_codeword]
        
        ################
        ## Channel
        ################
        y = self._channel(x, no)  # [batch size, num_symbols_per_codeword]
        
        ################
        ## Receiver
        ################
        llr = self._demapper(y, no)
        # Outer decoding
        b_hat = self._decoder(llr)
        return b, b_hat  # Ground truth and reconstructed information bits returned for BER/BLER computation

In [ ]:
class BaselineRayleigh(nn.Module):
    """Baseline over an i.i.d. Rayleigh flat-fading channel with perfect CSI."""

    def __init__(self):
        super().__init__()
        self._binary_source = BinarySource()
        self._encoder = LDPC5GEncoder(k, n, num_bits_per_symbol)
        constellation = Constellation("qam", num_bits_per_symbol)
        self.constellation = constellation
        self._mapper = Mapper(constellation=constellation)
        self._channel = AWGN()
        self._demapper = Demapper("app", constellation=constellation)
        self._decoder = LDPC5GDecoder(self._encoder, hard_out=True)

    def forward(self, batch_size, ebno_db):
        if ebno_db.dim() == 0:
            ebno_db = ebno_db.expand(batch_size)
        no = ebnodb2no(ebno_db, num_bits_per_symbol, coderate)
        no = expand_to_rank(no, 2)

        # Transmitter
        b = self._binary_source([batch_size, k])
        c = self._encoder(b)
        x = self._mapper(c)

        # Channel: Rayleigh flat fading, i.i.d. per symbol
        std = 0.5 ** 0.5
        h_r = torch.randn(x.shape, device=x.device, dtype=x.real.dtype) * std
        h_i = torch.randn(x.shape, device=x.device, dtype=x.real.dtype) * std
        h = torch.complex(h_r, h_i)
        y = self._channel(h * x, no)

        # Receiver: coherent equalization (perfect CSI)
        z = y / h
        no_eff = no / (h.real ** 2 + h.imag ** 2)

        llr = self._demapper(z, no_eff)
        b_hat = self._decoder(llr)
        return b, b_hat

## 5. Block-fading channel with pilot-based CSI

Frame structure: `h` is constant over `L = 25` symbols and i.i.d. across frames. The first `P`
symbols of each frame are unit-modulus pilots; the LS estimate is
`h_hat = (1/P) * sum(conj(x_p) * y_p)` with error variance `N0/P` (verified in Section 9).

Unit-modulus pilots are used deliberately so that the LS estimate reduces to a simple mean —
picking pilots from the 64-QAM constellation would break that.

In [ ]:
device = sionna.phy.config.device
print('device =', device)

In [ ]:
def make_rayleigh_block_frame(x_data, P, L, no, device, perfect_csi=False):
    """
    x_data: [B, num_data_symbols] complex
    no:     scalar or [B,1] noise power
    perfect_csi=True uses the true h (upper-bound reference); False uses the pilot estimate h_hat
    returns z, no_eff, h, h_hat, num_frames, pad
    """
    B, num_data = x_data.shape
    data_per_frame = L - P
    num_frames = (num_data + data_per_frame - 1) // data_per_frame

    # --- one independent h per frame ---
    std = 0.5 ** 0.5
    h = torch.complex(
        torch.randn(B, num_frames, 1, device=device) * std,
        torch.randn(B, num_frames, 1, device=device) * std,
    )  # [B, F, 1]

    # --- split data symbols into frames ---
    pad = num_frames * data_per_frame - num_data
    x_padded = F.pad(x_data, (0, pad))
    x_framed = x_padded.reshape(B, num_frames, data_per_frame)  # [B, F, data_per_frame]

    if not torch.is_tensor(no):
        no = torch.tensor(float(no), device=device)
    no = no.reshape(-1, 1, 1) if no.ndim >= 1 else no.reshape(1, 1, 1)  # [B,1,1] or [1,1,1] for broadcasting
    n_std = (no / 2.0).sqrt()

    def cn(shape):  # complex Gaussian noise CN(0, no)
        return torch.complex(torch.randn(shape, device=device) * n_std,
                             torch.randn(shape, device=device) * n_std)

    # --- pilot segment: x_p = 1+0j, same h, estimate h_hat ---
    x_pilot = torch.ones(B, num_frames, P, device=device, dtype=x_data.dtype)  # unit modulus
    y_pilot = h * x_pilot + cn((B, num_frames, P))
    # LS estimate: h_hat = (1/P) sum conj(x_p) y_p (mean for unit-modulus pilots)
    h_hat = (torch.conj(x_pilot) * y_pilot).mean(dim=2, keepdim=True)  # [B, F, 1]

    # --- data segment: through the true h ---
    y_framed = h * x_framed + cn((B, num_frames, data_per_frame))

    # --- equalize with h_hat (or true h for upper-bound reference) ---
    h_use = h if perfect_csi else h_hat
    z_framed = y_framed / h_use
    no_eff_framed = no / (h_use.real ** 2 + h_use.imag ** 2)
    no_eff_framed = no_eff_framed.expand(-1, -1, data_per_frame)

    # --- flatten to symbol stream, drop padding ---
    z = z_framed.reshape(B, -1)[:, :num_data]
    no_eff = no_eff_framed.reshape(B, -1)[:, :num_data]

    return z, no_eff, h, h_hat, num_frames, pad

In [ ]:
class E2ESystemPilotCSI(E2ESystemConventionalTraining):
    """Block-fading + pilot-estimated CSI. Inherits TX/RX, reuses AWGN weights."""

    def __init__(self, P, L, training=False, perfect_csi=False):
        super().__init__(training=training)
        self.P = P
        self.L = L
        self.perfect_csi = perfect_csi

    def forward(self, batch_size, ebno_db):
        self.constellation.points = torch.complex(self.points_r, self.points_i)
        if ebno_db.dim() == 0:
            ebno_db = ebno_db.expand(batch_size)

        # key: fold pilot overhead into the effective code rate
        coderate_eff = coderate * (self.L - self.P) / self.L
        no = ebnodb2no(ebno_db, num_bits_per_symbol, coderate_eff)
        no = expand_to_rank(no, 2)

        # transmitter
        if self._training:
            c = self._binary_source([batch_size, n])
        else:
            b = self._binary_source([batch_size, k])
            c = self._encoder(b)
        x = self._mapper(c)   # [B, 250]

        # block-fading channel + pilot estimation (reuse the verified helper)
        no_scalar = no[:, :1]  # [B,1]
        z, no_eff, h, h_hat, nf, pad = make_rayleigh_block_frame(
            x, P=self.P, L=self.L, no=no_scalar,
            device=x.device, perfect_csi=self.perfect_csi)

        # receiver
        llr = self._demapper(z, no_eff)
        llr = llr.reshape(batch_size, n)

        if self._training:
            return F.binary_cross_entropy_with_logits(llr, c)
        else:
            b_hat = self._decoder(llr)
            return b, b_hat

## 6. CNN demapper

1-D convolutions with a receptive field spanning neighbouring symbols inside a frame.
Input channels: `[Re(z), Im(z), Re(h_hat), Im(h_hat), log10(no_eff)]`.

In [ ]:
def make_frame_cnn(x_data, P, L, no, device, perfect_csi=False):
    """Return a frame-structured tensor for the CNN (not flattened).
    features: [B, num_frames, data_per_frame, C]  C=5: Re(z),Im(z),Re(h_hat),Im(h_hat),log10(no_eff)
    """
    B, num_data = x_data.shape
    dpf = L - P
    nf = (num_data + dpf - 1) // dpf
    std = 0.5 ** 0.5
    h = torch.complex(torch.randn(B, nf, 1, device=device) * std,
                      torch.randn(B, nf, 1, device=device) * std)
    pad = nf * dpf - num_data
    x_framed = F.pad(x_data, (0, pad)).reshape(B, nf, dpf)

    if not torch.is_tensor(no):
        no = torch.tensor(float(no), device=device)
    no = no.reshape(-1, 1, 1) if no.ndim >= 1 else no.reshape(1, 1, 1)
    n_std = (no / 2.0).sqrt()
    def cn(shape):
        return torch.complex(torch.randn(shape, device=device) * n_std,
                             torch.randn(shape, device=device) * n_std)

    x_pilot = torch.ones(B, nf, P, device=device, dtype=x_data.dtype)
    y_pilot = h * x_pilot + cn((B, nf, P))
    h_hat = (torch.conj(x_pilot) * y_pilot).mean(dim=2, keepdim=True)  # [B,nf,1]

    y_framed = h * x_framed + cn((B, nf, dpf))
    h_use = h if perfect_csi else h_hat
    z = y_framed / h_use                                    # [B,nf,dpf]
    no_eff = (no / (h_use.real**2 + h_use.imag**2))         # [B,nf,1]

    # assemble channels, broadcast h_hat and no_eff to each symbol
    h_use_b = h_use.expand(-1, -1, dpf)
    no_eff_b = no_eff.expand(-1, -1, dpf)
    feats = torch.stack([z.real, z.imag,
                         h_use_b.real, h_use_b.imag,
                         torch.log10(no_eff_b)], dim=-1)     # [B,nf,dpf,5]
    return feats, num_data, nf, pad, dpf

In [ ]:
class CNNDemapper(nn.Module):
    """In-frame 1D convolution; receptive field spans neighbouring symbols."""
    def __init__(self, in_ch=5):
        super().__init__()
        self.c1 = nn.Conv1d(in_ch, 64, kernel_size=3, padding=1)
        self.c2 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.c3 = nn.Conv1d(64, num_bits_per_symbol, kernel_size=1)

    def forward(self, feats):
        # feats [B, nf, dpf, C] -> [B*nf, C, dpf]
        B, nf, dpf, C = feats.shape
        x = feats.reshape(B * nf, dpf, C).permute(0, 2, 1)
        x = F.relu(self.c1(x))
        x = F.relu(self.c2(x))
        x = self.c3(x)                       # [B*nf, 6, dpf]
        x = x.permute(0, 2, 1).reshape(B, nf, dpf, num_bits_per_symbol)
        return x

class E2ESystemPilotCNN(E2ESystemConventionalTraining):
    def __init__(self, P, L, training=False, perfect_csi=False):
        super().__init__(training=training)
        self.P = P; self.L = L; self.perfect_csi = perfect_csi
        self._demapper = CNNDemapper(in_ch=5)   # override the MLP demapper

    def forward(self, batch_size, ebno_db):
        self.constellation.points = torch.complex(self.points_r, self.points_i)
        if ebno_db.dim() == 0:
            ebno_db = ebno_db.expand(batch_size)
        coderate_eff = coderate * (self.L - self.P) / self.L
        no = ebnodb2no(ebno_db, num_bits_per_symbol, coderate_eff)
        no = expand_to_rank(no, 2)

        if self._training:
            c = self._binary_source([batch_size, n])
        else:
            b = self._binary_source([batch_size, k])
            c = self._encoder(b)
        x = self._mapper(c)

        feats, num_data, nf, pad, dpf = make_frame_cnn(
            x, self.P, self.L, no[:, :1], x.device, self.perfect_csi)
        llr_framed = self._demapper(feats)               # [B,nf,dpf,6]
        llr = llr_framed.reshape(batch_size, -1, num_bits_per_symbol)[:, :num_data, :]
        llr = llr.reshape(batch_size, n)

        if self._training:
            return F.binary_cross_entropy_with_logits(llr, c)
        else:
            b_hat = self._decoder(llr)
            return b, b_hat

## 7. Uncoded system (control for Section 16)

Identical TX/RX, but **no LDPC**: raw bits to constellation to channel to demapper to hard
decision. Isolates how much of the final BER is due to channel coding.

In [ ]:
class E2ESystemPilotUncoded(E2ESystemPilotCSI):
    """Same TX/RX as E2ESystemPilotCSI, but NO LDPC:
    raw bits -> constellation -> channel -> demapper -> hard decision.
    Shows what BER looks like WITHOUT channel coding."""

    def forward(self, batch_size, ebno_db):
        self.constellation.points = torch.complex(self.points_r, self.points_i)
        if ebno_db.dim() == 0:
            ebno_db = ebno_db.expand(batch_size)

        # 无编码：能量归一化不含码率增益（coderate=1），但导频开销仍在
        coderate_uncoded = 1.0 * (self.L - self.P) / self.L
        no = ebnodb2no(ebno_db, num_bits_per_symbol, coderate_uncoded)
        no = expand_to_rank(no, 2)

        # 直接生成 n 个随机比特（p=0.5），全部是数据，无 LDPC
        b = self._binary_source([batch_size, n])
        x = self._mapper(b)

        z, no_eff, h, h_hat, nf, pad = make_rayleigh_block_frame(
            x, P=self.P, L=self.L, no=no[:, :1],
            device=x.device, perfect_csi=self.perfect_csi)

        llr = self._demapper(z, no_eff)
        llr = llr.reshape(batch_size, n)

        # 硬判决：LLR>0 → 1（无译码器兜底）
        b_hat = (llr > 0).to(b.dtype)
        return b, b_hat

# Part II — Experiments

Each section states a question and answers it with a measurement. Slow sections are flagged.

## 8. Train the autoencoder on AWGN  *(~4-5 min)*

Produces the AWGN weights that warm-start every later Rayleigh experiment. Final BCE ~0.286
(reference implementation reports 0.289).

In [ ]:
# Instantiate and train the end-to-end system
model_weights_path_conventional_training = 'awgn_autoencoder_weights_conventional_training'
model = E2ESystemConventionalTraining(training=True).to(device)
conventional_training(model)
save_weights(model, model_weights_path_conventional_training)

## 9. Validation: LS estimation accuracy

**Question:** is the channel estimator correct?
**Result:** measured MSE matches the theoretical `N0/P` to within 1e-3 for every P, and doubling
P halves the error — as predicted.

In [ ]:
# LS accuracy check: error should be ~ theory no/P
torch.manual_seed(0)
x_test = torch.randn(64, 250, dtype=torch.complex64, device=device)
no_test = 0.1
for P in [2, 4, 6, 8]:
    z, no_eff, h, h_hat, nf, pad = make_rayleigh_block_frame(
        x_test, P=P, L=25, no=no_test, device=device)
    err = (h - h_hat).abs().pow(2).mean().item()
    print(f'P={P}: MSE={err:.5f}   theory no/P={no_test/P:.5f}')

## 10. Validation: pilot-CSI forward pass

Confirms shapes and that the full chain runs.

In [ ]:
m = E2ESystemPilotCSI(P=4, L=25, training=False, perfect_csi=False).to(device)
load_weights(m, model_weights_path_conventional_training)
ebno = torch.tensor(12.0, device=device)
b, b_hat = m(64, ebno)
print('b:', b.shape, 'b_hat:', b_hat.shape)
print('raw BER:', (b != b_hat).float().mean().item())

## 11. Failure mode: the U-shaped BER curve  *(~15-20 min)*

**Question:** can AWGN-trained weights be used directly on Rayleigh?

**Answer: no.** BER *rises* at high SNR — non-physical, so something is wrong. This section
documents the failure; Section 12 fixes it.

**Diagnosis by bisection (11b):** the APP baseline on the *same* channel is perfectly monotonic,
while the neural demapper shows the U-shape even under perfect CSI. The fault is therefore
isolated to the demapper — not the channel model, not the LDPC decoder.

**Root cause — train/test distribution shift.** Under AWGN the network only ever saw
`log10(N0)` within `[-1.28, -0.88]`. In a deep fade, `z = y/h` divides by a near-zero number and
pushes inputs orders of magnitude outside that range. A ReLU network **extrapolates linearly**
out of distribution, and its unbounded final layer emits large, confident and *wrong* LLRs,
which LDPC reads as high confidence and propagates. Higher SNR sharpens the deep-fade contrast,
so the curve worsens as SNR grows.

### 11a. Sweep without retraining — the U-shape appears

In [ ]:
import numpy as np

ebno_dbs_pilot = np.arange(8.0, 18.1, 1.0)   # block-fading + pilot: waterfall shifts right
P_list = [2, 4, 6, 8]
BER_pilot = {}

for P in P_list:
    m = E2ESystemPilotCSI(P=P, L=25, training=False, perfect_csi=False).to(device)
    load_weights(m, model_weights_path_conventional_training)
    with torch.no_grad():
        ber, bler = sim_ber(m, ebno_dbs_pilot, batch_size=128,
                            num_target_block_errors=300, max_mc_iter=100)
    BER_pilot[f'P={P}'] = ber.cpu().numpy()
    print(f'--- P={P} done ---')

# add a perfect-CSI upper bound for reference
m = E2ESystemPilotCSI(P=4, L=25, training=False, perfect_csi=True).to(device)
load_weights(m, model_weights_path_conventional_training)
with torch.no_grad():
    ber, _ = sim_ber(m, ebno_dbs_pilot, batch_size=128,
                     num_target_block_errors=300, max_mc_iter=100)
BER_pilot['perfect CSI (P=4)'] = ber.cpu().numpy()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 6))
for label, ber in BER_pilot.items():
    ber = np.asarray(ber)
    mask = ber > 0
    ls = '--' if 'perfect' in label else '-'
    plt.semilogy(ebno_dbs_pilot[mask], ber[mask], marker='o', linestyle=ls, label=label)
plt.axhline(1e-2, color='gray', ls=':', label='target 1e-2')
plt.xlabel(r'$E_b/N_0$ (dB)')
plt.ylabel('BER')
plt.grid(which='both', alpha=0.4)
plt.legend()
plt.title('Block-fading + pilot CSI: BER vs pilot count P (L=25)')
plt.tight_layout()
plt.savefig('ber_vs_pilot.png', dpi=150)

### 11b. Bisection — neural demapper (perfect CSI) vs APP baseline

In [ ]:
import numpy as np
for ebno in [10.0, 14.0, 18.0]:
    m = E2ESystemPilotCSI(P=4, L=25, training=False, perfect_csi=True).to(device)
    load_weights(m, model_weights_path_conventional_training)
    with torch.no_grad():
        ber, bler = sim_ber(m, np.array([ebno]), batch_size=256,
                            num_target_block_errors=500, max_mc_iter=200)
    print(f"ebno={ebno}: BER={ber.item():.4e}")

In [ ]:
for ebno in [10.0, 14.0, 18.0]:
    m = BaselineRayleigh().to(device)
    with torch.no_grad():
        ber, _ = sim_ber(m, np.array([ebno]), batch_size=256,
                        num_target_block_errors=500, max_mc_iter=200)
    print(f"Baseline ebno={ebno}: BER={ber.item():.4e}")

## 12. The fix: retrain on the Rayleigh channel

Variables are isolated: retrain first under **perfect CSI** (proves the demapper is fixable),
then under **pilot estimation** (the realistic case).

### 12a. Perfect CSI — retrain, verify the U-shape is gone

In [ ]:
# retrain on block-fading + perfect CSI (no pilots yet, isolate variables)
model_rayleigh = E2ESystemPilotCSI(P=4, L=25, training=True, perfect_csi=True).to(device)
load_weights(model_rayleigh, model_weights_path_conventional_training)  # warm-start from AWGN weights

# retrain (SNR range aligned to the block-fading waterfall)
import torch
optimizer = torch.optim.Adam(model_rayleigh.parameters(), lr=1e-3)
for it in range(3000):
    ebno_db = torch.empty(128, device=device).uniform_(8.0, 16.0)  # align to block-fading waterfall
    optimizer.zero_grad()
    loss = model_rayleigh(128, ebno_db)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_rayleigh.parameters(), 1.0)  # guard against deep-fade gradient explosion
    optimizer.step()
    if it % 500 == 0:
        print(f"it {it}: BCE={loss.item():.4f}")

save_weights(model_rayleigh, 'rayleigh_perfect_csi_weights')

In [ ]:
import numpy as np

print("=== after retrain (rayleigh_perfect_csi_weights) ===")
for ebno in [10.0, 14.0, 18.0]:
    m = E2ESystemPilotCSI(P=4, L=25, training=False, perfect_csi=True).to(device)
    load_weights(m, 'rayleigh_perfect_csi_weights')       # <- use new weights
    with torch.no_grad():
        ber, bler = sim_ber(m, np.array([ebno]), batch_size=256,
                            num_target_block_errors=500, max_mc_iter=200)
    print(f"ebno={ebno}: BER={ber.item():.4e}")

### 12b. Estimated CSI — retrain, verify the target is met

Result: **3.5e-4 at 14 dB**, past the 1e-2 target with margin, under *estimated* CSI — the
condition that matters for feasibility, not the perfect-CSI upper bound.
Cost of estimation, quantified: converged BCE 0.207 (perfect CSI) vs 0.230 (P = 4 pilots).

In [ ]:
# final form: block-fading + pilot-estimated CSI, retrain
model_pilot = E2ESystemPilotCSI(P=4, L=25, training=True, perfect_csi=False).to(device)
load_weights(model_pilot, model_weights_path_conventional_training)  # warm-start from AWGN

optimizer = torch.optim.Adam(model_pilot.parameters(), lr=1e-3)
for it in range(3000):
    ebno_db = torch.empty(128, device=device).uniform_(8.0, 16.0)
    optimizer.zero_grad()
    loss = model_pilot(128, ebno_db)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_pilot.parameters(), 1.0)
    optimizer.step()
    if it % 500 == 0:
        print(f"it {it}: BCE={loss.item():.4f}")

save_weights(model_pilot, 'rayleigh_pilot_csi_weights')

In [ ]:
import numpy as np

print("=== pilot estimation + retrain (rayleigh_pilot_csi_weights) ===")
for ebno in [10.0, 14.0, 18.0]:
    m = E2ESystemPilotCSI(P=4, L=25, training=False, perfect_csi=False).to(device)
    load_weights(m, 'rayleigh_pilot_csi_weights')
    with torch.no_grad():
        ber, bler = sim_ber(m, np.array([ebno]), batch_size=256,
                            num_target_block_errors=500, max_mc_iter=200)
    print(f"ebno={ebno}: BER={ber.item():.4e}")

## 13. Pilot sweep: how many pilots?  *(~20-25 min)*

**Question:** what is the optimal pilot count P?

**Answer: it depends on the operating SNR.** Two opposing forces — more pilots improve the
estimate (`N0/P`) but cost rate (`(L-P)/L`). The curves cross near 11 dB: below it fewer pilots
win, above it more pilots win. There is no single best P, only a trade-off.

In [ ]:
import numpy as np

ebno_dbs_final = np.arange(8.0, 16.1, 1.0)
P_list = [2, 4, 6, 8]
BER_final = {}
weights_by_P = {}

# 1) retrain each P separately (warm-start from AWGN)
for P in P_list:
    print(f"===== retrain P={P} =====")
    mt = E2ESystemPilotCSI(P=P, L=25, training=True, perfect_csi=False).to(device)
    load_weights(mt, model_weights_path_conventional_training)
    opt = torch.optim.Adam(mt.parameters(), lr=1e-3)
    for it in range(2000):
        ebno_db = torch.empty(128, device=device).uniform_(8.0, 16.0)
        opt.zero_grad()
        loss = mt(128, ebno_db)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(mt.parameters(), 1.0)
        opt.step()
    wpath = f'rayleigh_pilot_weights_P{P}'
    save_weights(mt, wpath)
    weights_by_P[P] = wpath
    print(f"  P={P} trained, BCE={loss.item():.4f}")

# 2) evaluate BER for each P
for P in P_list:
    me = E2ESystemPilotCSI(P=P, L=25, training=False, perfect_csi=False).to(device)
    load_weights(me, weights_by_P[P])
    with torch.no_grad():
        ber, _ = sim_ber(me, ebno_dbs_final, batch_size=128,
                         num_target_block_errors=300, max_mc_iter=100)
    BER_final[f'P={P}'] = ber.cpu().numpy()
    print(f"--- P={P} evaluated ---")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(9, 6))
for label, ber in BER_final.items():
    ber = np.asarray(ber)
    mask = ber > 0
    plt.semilogy(ebno_dbs_final[mask], ber[mask], marker='o', label=label)
plt.axhline(1e-2, color='gray', ls=':', label='target 1e-2')
plt.xlabel(r'$E_b/N_0$ (dB)')
plt.ylabel('BER')
plt.grid(which='both', alpha=0.4)
plt.legend()
plt.title('Retrained on Rayleigh + pilot CSI: BER vs pilot count P (L=25)')
plt.tight_layout()
plt.savefig('ber_vs_P_retrained.png', dpi=150)
plt.show()

## 14. CNN vs MLP demapper

**Question:** does a receptive field over neighbouring symbols help?

**Answer: no.** The curves nearly overlap, with the MLP marginally better at high SNR. After
equalisation, flat block fading reduces to a *per-symbol* AWGN channel — neighbours share only
`h_hat`, which is already an input channel. The optimal demapper here is per-symbol, which the
MLP already matches. A CNN's advantage would require a frequency-selective (multipath / OFDM)
channel — which makes OFDM the natural next step.

In [ ]:
mc = E2ESystemPilotCNN(P=2, L=25, training=False, perfect_csi=False).to(device)
ebno = torch.tensor(12.0, device=device)
b, b_hat = mc(64, ebno)
print("b:", b.shape, "b_hat:", b_hat.shape)   # expect [64,750] [64,750]
print("forward pass OK")

In [ ]:
# train CNN demapper (P=2 optimal, from scratch)
model_cnn = E2ESystemPilotCNN(P=2, L=25, training=True, perfect_csi=False).to(device)

optimizer = torch.optim.Adam(model_cnn.parameters(), lr=1e-3)
for it in range(4000):                      # CNN trains from scratch, give it more steps than MLP
    ebno_db = torch.empty(128, device=device).uniform_(8.0, 16.0)
    optimizer.zero_grad()
    loss = model_cnn(128, ebno_db)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_cnn.parameters(), 1.0)
    optimizer.step()
    if it % 500 == 0:
        print(f"it {it}: BCE={loss.item():.4f}")

save_weights(model_cnn, 'rayleigh_cnn_P2_weights')

In [ ]:
import numpy as np
print("=== CNN P=2 ===")
for ebno in [10.0, 14.0, 18.0]:
    m = E2ESystemPilotCNN(P=2, L=25, training=False, perfect_csi=False).to(device)
    load_weights(m, 'rayleigh_cnn_P2_weights')
    with torch.no_grad():
        ber, _ = sim_ber(m, np.array([ebno]), batch_size=256,
                         num_target_block_errors=500, max_mc_iter=200)
    print(f"ebno={ebno}: BER={ber.item():.4e}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ebno_cmp = np.arange(8.0, 16.1, 1.0)

# MLP (P=2) -- use the previously saved weights
m_mlp = E2ESystemPilotCSI(P=2, L=25, training=False, perfect_csi=False).to(device)
load_weights(m_mlp, weights_by_P[2])          # P=2 weights saved during the pilot sweep
with torch.no_grad():
    ber_mlp, _ = sim_ber(m_mlp, ebno_cmp, batch_size=128,
                         num_target_block_errors=300, max_mc_iter=100)

# CNN (P=2)
m_cnn = E2ESystemPilotCNN(P=2, L=25, training=False, perfect_csi=False).to(device)
load_weights(m_cnn, 'rayleigh_cnn_P2_weights')
with torch.no_grad():
    ber_cnn, _ = sim_ber(m_cnn, ebno_cmp, batch_size=128,
                         num_target_block_errors=300, max_mc_iter=100)

ber_mlp = ber_mlp.cpu().numpy()
ber_cnn = ber_cnn.cpu().numpy()

plt.figure(figsize=(9, 6))
mask = ber_mlp > 0
plt.semilogy(ebno_cmp[mask], ber_mlp[mask], 'o-', label='MLP demapper (P=2)')
mask = ber_cnn > 0
plt.semilogy(ebno_cmp[mask], ber_cnn[mask], 's--', label='CNN demapper (P=2)')
plt.axhline(1e-2, color='gray', ls=':', label='target 1e-2')
plt.xlabel(r'$E_b/N_0$ (dB)')
plt.ylabel('BER')
plt.grid(which='both', alpha=0.4)
plt.legend()
plt.title('MLP vs CNN demapper, block-fading + pilot CSI (P=2, L=25)')
plt.tight_layout()
plt.savefig('ber_mlp_vs_cnn.png', dpi=150)
plt.show()

## 15. Capacity study: is 128 hidden units too large?  *(~12 min)*

**Question:** the tutorial defaults to 128 hidden units — is that overkill?

**Answer: yes.** Width 64 is indistinguishable from 128; width 32 (1/13 the parameters) is
nearly equal; only width 16 degrades. This also confirms that the Rayleigh failure of Section 11
was never a capacity problem — capacity was already in excess, so the root cause had to lie in
the input distribution.

In [ ]:
# ============================================================
# Width sweep: is 128 hidden units overkill?
# Train the SAME task (AWGN) at width 16/32/64/128, then
# compare converged BCE and BER at a few SNR points.
# ============================================================
import numpy as np
import copy

widths = [16, 32, 64, 128]
sweep_iters = 5000          # 一半于原版10000，足够看收敛趋势且省时
eval_ebno = np.array([5.0, 6.0, 7.0])   # AWGN 瀑布区三点
results = {}

for w in widths:
    print(f"===== width={w} =====")
    torch.manual_seed(42)                      # ← 同一初始化种子，公平对比
    model = E2ESystemConventionalTraining(training=True).to(device)
    # 换上指定宽度的 demapper（星座参数保持原版初始化）
    model._demapper = NeuralDemapper(width=w).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for it in range(sweep_iters):
        ebno_db = torch.empty(128, device=device).uniform_(ebno_db_min, ebno_db_max)
        opt.zero_grad()
        loss = model(128, ebno_db)
        loss.backward()
        opt.step()
    final_bce = loss.item()

    # 参数量统计（只算 demapper）
    n_params = sum(p.numel() for p in model._demapper.parameters())

    # 评估 BER
    wpath = f'awgn_width{w}_weights'
    save_weights(model, wpath)
    me = E2ESystemConventionalTraining(training=False).to(device)
    me._demapper = NeuralDemapper(width=w).to(device)
    load_weights(me, wpath)
    with torch.no_grad():
        ber, _ = sim_ber(me, eval_ebno, batch_size=128,
                         num_target_block_errors=300, max_mc_iter=100)
    results[w] = {'bce': final_bce, 'params': n_params,
                  'ber': ber.cpu().numpy()}
    print(f"  width={w}: params={n_params}, BCE={final_bce:.4f}")

# ============ 汇总表 ============
print("\n================= SUMMARY =================")
print(f"{'width':>6} | {'params':>7} | {'BCE':>7} | {'BER@5dB':>10} | {'BER@6dB':>10} | {'BER@7dB':>10}")
for w in widths:
    r = results[w]
    print(f"{w:>6} | {r['params']:>7} | {r['bce']:.4f} | "
          f"{r['ber'][0]:.3e} | {r['ber'][1]:.3e} | {r['ber'][2]:.3e}")

## 16. Where does 1e-5 come from? Coded vs uncoded  *(~15 min)*

**Question:** 1e-5 on Rayleigh with estimated CSI looks too good — is channel state leaking
into the transmitter?

**Answer: no leak — it is LDPC coding gain.** The uncoded system, using the *same* demapper
weights and the *same* channel, never drops below 2e-2 even at 20 dB. The entire gap between
the two curves is coding gain. The commonly quoted 1e-2/1e-3 figure for estimated CSI refers to
*uncoded* transmission — exactly what the red curve reproduces.

Note: the uncoded curve has BLER = 1 throughout (a 1500-bit block with 2% BER always contains
errors), so `sim_ber` terminates early and its sample size is smaller. This does not affect a
conclusion drawn across three orders of magnitude.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ebno_wide = np.arange(4.0, 20.1, 2.0)

# 1) 未编码 + 神经 demapper(P=4 重训权重)
mu = E2ESystemPilotUncoded(P=4, L=25, training=False, perfect_csi=False).to(device)
load_weights(mu, 'rayleigh_pilot_weights_P4')
with torch.no_grad():
    ber_unc, _ = sim_ber(mu, ebno_wide, batch_size=128,
                         num_target_block_errors=300, max_mc_iter=100)

# 2) 带 LDPC 的完整系统（同权重，同信道）
mc = E2ESystemPilotCSI(P=4, L=25, training=False, perfect_csi=False).to(device)
load_weights(mc, 'rayleigh_pilot_weights_P4')
with torch.no_grad():
    ber_cod, _ = sim_ber(mc, ebno_wide, batch_size=128,
                         num_target_block_errors=300, max_mc_iter=100)

# ============ 画图 ============
plt.figure(figsize=(9, 6))
b = ber_unc.cpu().numpy(); m = b > 0
plt.semilogy(ebno_wide[m], b[m], 's--', color='C3',
             label='Uncoded (no LDPC), pilot CSI P=4')
b = ber_cod.cpu().numpy(); m = b > 0
plt.semilogy(ebno_wide[m], b[m], 'o-', color='C0',
             label='With LDPC (rate 0.5), pilot CSI P=4')
plt.axhspan(1e-3, 1e-2, color='gray', alpha=0.15,
            label="advisor's expected range (uncoded)")
plt.xlabel(r'$E_b/N_0$ (dB)')
plt.ylabel('BER')
plt.grid(which='both', alpha=0.4)
plt.legend()
plt.title('Where does 1e-5 come from? Coding gain of LDPC (Rayleigh, estimated CSI)')
plt.tight_layout()
plt.savefig('ber_coded_vs_uncoded.png', dpi=150)
plt.show()

## 17. Does the training-SNR strategy matter?  *(~20 min)*

**Question:** train at a fixed mid SNR, or over a range?

**Answer: almost no difference.** Fixed 15 dB, uniform 8-16 dB and uniform 0-20 dB give
overlapping curves across the whole 0-20 dB evaluation range.

**Why:** `log10(N0)` is an *explicit input*, so the network learns a family of functions
conditioned on noise power rather than a single fixed-SNR demapper. Rayleigh deep fades also
spread the *effective* SNR across orders of magnitude even when the nominal SNR is fixed. This
is the mirror image of Section 11: there the network had never seen the range; here it always
has.

*Caveat: this conclusion depends on N0 being an input. A design without it would be far more
sensitive to the training SNR.*

In [ ]:
# ============================================================
# SNR training strategy comparison
# A: fixed 15 dB | B: uniform 8-16 dB (current) | C: uniform 0-20 dB
# All evaluated over the full 0-20 dB range.
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

strategies = {
    'fixed 15 dB':      ('fixed', 15.0, 15.0),
    'uniform 8-16 dB':  ('range',  8.0, 16.0),
    'uniform 0-20 dB':  ('range',  0.0, 20.0),
}
ebno_eval = np.arange(0.0, 20.1, 2.0)
BER_snr = {}

for name, (kind, lo, hi) in strategies.items():
    print(f"===== training: {name} =====")
    torch.manual_seed(42)
    mt = E2ESystemPilotCSI(P=4, L=25, training=True, perfect_csi=False).to(device)
    load_weights(mt, model_weights_path_conventional_training)   # AWGN 热启动
    opt = torch.optim.Adam(mt.parameters(), lr=1e-3)
    for it in range(3000):
        if kind == 'fixed':
            ebno_db = torch.full((128,), lo, device=device)
        else:
            ebno_db = torch.empty(128, device=device).uniform_(lo, hi)
        opt.zero_grad()
        loss = mt(128, ebno_db)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(mt.parameters(), 1.0)
        opt.step()
    wpath = f"snrstrat_{name.replace(' ','_').replace('-','_')}"
    save_weights(mt, wpath)
    print(f"  BCE={loss.item():.4f}")

    me = E2ESystemPilotCSI(P=4, L=25, training=False, perfect_csi=False).to(device)
    load_weights(me, wpath)
    with torch.no_grad():
        ber, _ = sim_ber(me, ebno_eval, batch_size=128,
                         num_target_block_errors=300, max_mc_iter=100)
    BER_snr[name] = ber.cpu().numpy()
    print(f"--- {name} evaluated ---")

# ============ 画图 ============
plt.figure(figsize=(9, 6))
for name, ber in BER_snr.items():
    ber = np.asarray(ber); m = ber > 0
    plt.semilogy(ebno_eval[m], ber[m], marker='o', label=f'trained: {name}')
plt.axvline(15.0, color='gray', ls=':', alpha=0.6)
plt.text(15.1, 2e-1, 'fixed-15dB\ntraining point', fontsize=8, color='gray')
plt.xlabel(r'$E_b/N_0$ (dB)')
plt.ylabel('BER')
plt.grid(which='both', alpha=0.4)
plt.legend()
plt.title('SNR training strategy: generalisation over 0-20 dB (Rayleigh, pilot CSI P=4)')
plt.tight_layout()
plt.savefig('ber_snr_strategy.png', dpi=150)
plt.show()

## Next steps

1. **OFDM extension** — prerequisite for multipath channels (TDL / CDL, `CIRDataset`) and the
   regime where a CNN demapper could actually pay off.
2. **Semi-blind estimation** — let the demapper refine `h_hat` using data symbols, not pilots alone.
3. **Remaining ablations** — activation variants, dropout / normalisation. Huber loss is expected
   *not* to help the demapper (BCE's gradient w.r.t. the logit is already bounded in [-1, 1]);
   it would be the right tool only for a dedicated channel-estimation head regressing on `h`.